<a href="https://colab.research.google.com/github/DylanNajib-Biotech/GloPPI-Ranker/blob/main/GloPPI_Ranker_v1.0_20-12-2025.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## **GloPPI Ranker v1.0**

**"Global *(alignment)* Pairwise Percent Identity" Ranker**


GloPPI Ranker applies Needleman-Wunsch alignment to quantify mutation types of multiple DNA sequences and their percent identity relative to a reference sequence.

Following the formula:

```
Percent Identity = n(Alignmatch)/n(DNA[1])*100 %
```
Where;
- n(Alignmatch): #matching bases after global alignment
- n(DNA[1]): #bases of reference sequence

\
**Quick Example**\
Input these (in order) to see the functions of GloPPI Ranker quickly;
>#Sequences: `7`\
>DNA[1]: `ATGCTAGCTAGGCTACGATCGTACGATGCTAGCTAGGCTACGATCGTACGAT`\
>DNA[2]: `ATGCTAGCTAGGCTACGATCGTACGATGCTAGCTAGGATACGATCGTACGAT`\
>DNA[3]: `ATGCTAGCTAGGCTACGATCGTACGATGCTAGCTAGGCTACGATC`\
>DNA[4]: `ATGCTAGCTACCGTACGATCGTACGATGCTAGCTAGGCTACGATCGTACGA`\
>DNA[5]: `ATGCTAGCTAGGCTACGATCGTACGATGCTAGCTAGG`\
>DNA[6]: `ATGCTAGCTAGGCTACGATCGTACGATGCTAGCTAGGGTACGATCGTACGAT`\
>DNA[7]: `TACGATCGATCCGATGCTAGCATGCTACGATCGATCCGATGCTAGCATGCTA`\
>\
>*P.S. - DNA[7] is a theoretical 0% percent identity sequence, IF unaligned*

\
**Current Limitations**
- Mutation positions unknown (base #?)
- Tied percent identities are not acknowledged

..

**Extra Plans**
- Resulting amino acid sequence
- Amino acids that were removed or changed
- Resulting protein (if possible)
- Likeliest mutation type
- Synonymous mutation
- Nonsense mutation
- Frameshift variations

In [ ]:
# @title
# GloPPI Ranker - Dylan Najib

## IMPORTS
import matplotlib.pyplot as plt;from IPython.display import clear_output
import time;import pandas as pd;from IPython.display import display;import ipywidgets as widgets

# Code
## GloPPI
def gloppi():
  clear_output(wait=True)
  dnalist=[]
  while True:
    seqanswer=input("#Sequences: ")
    try:
      nlist=int(seqanswer)
      if nlist <= 1:
        print("\n#Sequences = >1 only pls.")
        time.sleep(3.5);clear_output()
        continue
      break
    except ValueError:
      print("Integers only pls.")
      time.sleep(3.5);clear_output()
  print(" \nWhere DNA[1] = Reference Sequence ; n = #bases,\n")
  dna1=input("DNA[1]: ")
  print(" ")

## If >n(DNA[1]) or <n(DNA[1]):
### Penalty System (Needleman-Wunsch) - S = score matrix, so S[i][j] = stores best alignment score
  def nwsystem(paramdna1,paramdna,same=1,notsame=-1,gap=-1):
    n,w=len(paramdna1),len(paramdna)
    S=[[0]*(w+1) for _ in range(n+1)]
    for i in range(1,n+1):
      S[i][0]=S[i-1][0]+gap
    for j in range(1,w+1):
      S[0][j]=S[0][j-1]+gap
    for i in range(1,n+1):
      for j in range(1,w+1):
        diagonal=S[i-1][j-1]+(same if paramdna1[i-1]==paramdna[j-1] else notsame)
        up=S[i-1][j]+gap
        left=S[i][j-1]+gap
        S[i][j]=max(diagonal,up,left)
    i,j=n,w
    samesies=0
    errsubs=0
    errins=0
    errdels=0
    align_profile=[]
    while i>0 or j>0:
      if i>0 and j>0 and S[i][j]==S[i-1][j-1]+(same if paramdna1[i-1]==paramdna[j-1] else notsame):
        if paramdna1[i-1]==paramdna[j-1]:
          samesies += 1
          align_profile.append("samesies")
        else:
          errsubs += 1
          align_profile.append("errsub")
        i -= 1
        j -= 1
      elif i>0 and S[i][j]==S[i-1][j]+gap:
        errdels += 1
        align_profile.append("errdel")
        i -= 1
      else:
        errins += 1
        align_profile.append("errins")
        j -= 1
    align_profile.reverse()
    return samesies,errsubs,errdels,errins,align_profile,samesies/len(paramdna1)*100

## COLLECTOR
  results=[]
  for i in range(nlist-1):
    dna=input(f"DNA[{i+2}]: ")
    print(" ")
    matching,errsubs,errdels,errins,align_profile,percentidentity=nwsystem(dna1,dna)
    dnalist.append((i+2,dna,percentidentity))
    results.append({
        "Index":f"DNA[{i+2}]",
        "Sequence":dna,
        "n(DNA)":len(dna),
        "n(Match)":int(matching),
        "#Sub(s)":int(errsubs),
        "#Del(s)":int(errdels),
        "#Ins(s)":int(errins),
        "FULL %Identity":f"{percentidentity}%",
        "2 DP %Identity":f"{round(percentidentity,2)}%"
        })
  results.insert(0,{
      "Index":"DNA[1]",
      "Sequence":dna1,
      "n(DNA)": len(dna1),
      "n(Match)":" ",
      "#Sub(s)":" ",
      "#Del(s)":" ",
      "#Ins(s)":" ",
      "FULL %Identity":" ",
      "2 DP %Identity":" ",
  })
  dnalist.insert(0,(1,dna1,100.0))

## RESULS
  dnalist.sort(key=lambda x: x[2],reverse=True)
### TABLE
  df = pd.DataFrame(results,columns=[
      "Index","Sequence","n(DNA)","n(Match)",
      "#Sub(s)","#Del(s)","#Ins(s)",
      "FULL %Identity","2 DP %Identity"
  ])
  df.set_index("Index",inplace=True)
  display(df)
  print(" ")
### CHART
  if nlist > 2:
    labels=[f"DNA[{idx}]" for idx,dna,sim in dnalist]
    sims=[sim for idx,dna,sim in dnalist]
    plt.figure(figsize=(10,6))
    bars=plt.barh(labels,sims)
    plt.xlabel("Percent Identity (%)");plt.ylabel("Sequences")
    plt.xlim(0,100)
    plt.grid(axis="x",linestyle="--",alpha=0.4)
    plt.gca().invert_yaxis() #### Reverse ranking for better visibility
    for bar,sim in zip(bars,sims):
      plt.annotate(
        f"{sim:.2f}%",
        xy=(bar.get_width(),bar.get_y()+bar.get_height()/2),
        xytext=(3,0),
        textcoords="offset points",
        ha="left",
        va="center"
      )
    plt.tight_layout()
    plt.show()
    print("\nDon't forget to screenshot!\n")

gloppi()